<a href="https://colab.research.google.com/github/agrim77com-boop/2P13--DS-LAB/blob/main/UCS420_LA9_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [ ]:
ROLL_NO = 1024160088          # <-- your full roll number
NAME = "Agrim Bhatt"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [5]:
# Q1
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from textblob import TextBlob

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
tokens = [word.lower() for word in word_tokenize(text) if word.isalpha()]
types = set(tokens)

print("Tokens:", tokens)
print("Total tokens:", len(tokens))
print("Types:", sorted(types))
print("Total types:", len(types))

NameError: name 'text' is not defined

**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:*  The final tokens such as `n't`, `'s`, `ca`, and `wo` are not complete words. They come from contractions in the original tickets, such as “isn't,” “it's,” “can't,” and “won't.” During NLTK tokenization, contractions are split into separate tokens. Since the preprocessing pipeline removes punctuation and stopwords but does not combine these fragments back into complete words, some fragments remain in the final tokens.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

# Q2
import re

raw_tickets = df[text_col].tolist()

patterns = {
    "Long words": r"\b[A-Za-z]+\b(?=\s|[^\w]|$)",
    "Numbers": r"(?<![\w.])\d+(?:\.\d+)*(?![\w.])",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words starting with vowels": r"\b[aeiouAEIOU][a-zA-Z]*\b"
}

def extract_numbers(text):
    return re.findall(patterns["Numbers"], text)

def extract_long_words(text):
    return [
        word for word in re.findall(r"\b[A-Za-z]+\b", text)
        if len(word) > 5
    ]

def extract_capitalized(text):
    return re.findall(patterns["Capitalized words"], text)

def extract_vowels(text):
    return re.findall(patterns["Words starting with vowels"], text)

def clean_text(text):
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        "<EMAIL>", text
    )
    text = re.sub(
        r"(?:https?://|www\.)[^\s<>]+",
        "<URL>", text
    )
    text = re.sub(
        r"(?<!\w)(?:\+\d{1,3}[- ]?)?(?:\d{3,5}[- ]?)?\d{5}[- ]?\d{4}(?!\w)",
        "<PHONE>", text
    )
    return text

for i, ticket in enumerate(raw_tickets, 1):
    print(f"\nTicket {i}")
    print("Long words:", extract_long_words(ticket))
    print("Numbers:", extract_numbers(ticket))
    print("Capitalized words:", extract_capitalized(ticket))
    print("Vowel-starting words:", extract_vowels(ticket))
    print("Cleaned:", clean_text(ticket))

print("\nTEST_LINES")
for line in TEST_LINES:
    print("\nOriginal:", line)
    print("Long words:", extract_long_words(line))
    print("Numbers:", extract_numbers(line))
    print("Capitalized words:", extract_capitalized(line))
    print("Vowel-starting words:", extract_vowels(line))
    print("Cleaned:", clean_text(line))

capitalized = [
    word for ticket in raw_tickets
    for word in extract_capitalized(ticket)
]

sentence_initial = []
for ticket in raw_tickets:
    for sentence in re.split(r"(?<=[.!?])\s+", ticket):
        match = re.search(r"\b[A-Z][a-zA-Z]*\b", sentence)
        if match:
            sentence_initial.append(match.group())

print(
    "\nCapitalized words that may be sentence-initial:",
    len(sentence_initial)
)

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:*  Many of the capitalized words are capitalized only because they appear at the beginning of a sentence. Proper nouns, names, and acronyms may also be capitalized. The exact number depends on the generated tickets, so it can be determined by comparing the capitalized words with their positions in the original sentences.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:

# Q3
from nltk.tokenize import RegexpTokenizer, word_tokenize

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

pattern = (
    r"https?://[^\s<>]+|www\.[^\s<>]+|"
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|"
    r"\+\d{1,3}(?:[- ]\d+)+|\b\d{3,5}-\d{3}-\d{4}\b|"
    r"\b\d+(?:\.\d+)+\b|"
    r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b|"
    r"\b[A-Za-z]+(?:n't|'[A-Za-z]+)?\b|"
    r"\d+|[^\w\s]"
)

tokenizer = RegexpTokenizer(pattern)

def my_tokenize(text):
    return tokenizer.tokenize(text)

for i, line in enumerate(TEST_LINES, 1):
    print(f"\nTEST_LINES {i}")
    print("Custom:", my_tokenize(line))
    print("NLTK:", word_tokenize(line))

different_tickets = 0

for ticket in df[text_col].tolist():
    if my_tokenize(ticket) != word_tokenize(ticket):
        different_tickets += 1

print(
    "\nTickets tokenized differently:",
    different_tickets,
    "out of",
    len(df)
)

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:*

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
# Q4
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer(r"(ing|ed|es|s)$")
lemmatizer = WordNetLemmatizer()

unique_final_tokens = sorted(set(
    token
    for ticket in raw_tickets
    for token in process_ticket(ticket)["final"]
))

valid_words = [
    word for word in unique_final_tokens
    if re.fullmatch(r"[a-zA-Z]+", word)
]

tagged_words = pos_tag(valid_words)

pos_map = {
    "J": "a",
    "V": "v",
    "N": "n",
    "R": "r"
}

rows = []

for word, tag in tagged_words:
    pos = pos_map.get(tag[0], "n")

    porter_word = porter.stem(word)
    lancaster_word = lancaster.stem(word)
    regexp_word = regexp_stemmer.stem(word)
    lemma_word = lemmatizer.lemmatize(word, pos=pos)

    if len({
        porter_word,
        lancaster_word,
        regexp_word,
        lemma_word
    }) > 1:
        rows.append({
            "Word": word,
            "POS": tag,
            "Porter": porter_word,
            "Lancaster": lancaster_word,
            "Regexp": regexp_word,
            "Lemmatizer": lemma_word
        })

q4_table = pd.DataFrame(rows)
display(q4_table)

print("Over-stemming example candidates:")
display(q4_table[
    q4_table.apply(
        lambda r: len({
            r["Porter"], r["Lancaster"], r["Regexp"]
        }) > 1,
        axis=1
    )
].head(10))

print("Under-stemming example candidates:")
display(q4_table[
    q4_table["Lemmatizer"] != q4_table["Porter"]
].head(10))

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:*
Over-stemming example: “studies” may be reduced to “studi,” which is not a complete English word.

Under-stemming example: “playing” and “played” may remain different forms if the stemming method fails to reduce them to the same root.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Q5
from nltk.util import ngrams
from collections import Counter
import matplotlib.pyplot as plt
import math

S1 = []
S2 = []
S3 = []

for ticket in raw_tickets:
    result = process_ticket(ticket)
    S1.extend(result["tokens"])
    S2.extend(
        token for token in result["tokens"]
        if not is_punct(token)
    )
    S3.extend(result["final"])

S4 = [porter.stem(token) for token in S3]

def corpus_stats(name, corpus):
    print(f"\n{name}")
    print("Tokens:", len(corpus))
    print("Unique tokens:", len(set(corpus)))
    print(
        "TTR:",
        len(set(corpus)) / len(corpus) if corpus else 0
    )

corpus_stats("S1", S1)
corpus_stats("S2", S2)
corpus_stats("S3", S3)
corpus_stats("S4", S4)

freq_s1 = Counter(S1)
freq_s3 = Counter(S3)

print("\nTop 10 words in S1:")
print(freq_s1.most_common(10))

print("\nTop 10 words in S3:")
print(freq_s3.most_common(10))

ranked_s3 = freq_s3.most_common()
ranks = range(1, len(ranked_s3) + 1)
frequencies = [frequency for _, frequency in ranked_s3]

plt.figure(figsize=(8, 5))
plt.plot(
    [math.log(rank) for rank in ranks],
    [math.log(freq) for freq in frequencies],
    marker="o"
)
plt.xlabel("log(Rank)")
plt.ylabel("log(Frequency)")
plt.title("Zipf's Law for S3")
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nTop 5 bigrams in S1:")
print(Counter(ngrams(S1, 2)).most_common(5))

print("\nTop 5 bigrams in S3:")
print(Counter(ngrams(S3, 2)).most_common(5))

print("\nTTR comparison:")
print("S1:", len(set(S1)) / len(S1) if S1 else 0)
print("S4:", len(set(S4)) / len(S4) if S4 else 0)

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:* TTR changes from S1 to S4 because S1 contains all tokens, including punctuation and stopwords, whereas S4 contains the final tokens after preprocessing and Porter stemming. Removing punctuation and stopwords reduces the total number of tokens, while stemming combines different word forms into the same root. As a result, the number of unique tokens and the total number of tokens change, causing the Type-Token Ratio (TTR) to change.